# 🎵 Messy Mashup - Music Genre Classification
## DL-23f2002518-notebook-t12026

**Student:** Sachin Kumar Ray (23f2002518)

### Models:
1. **EfficientNet-B0** (CNN - Pretrained backbone, custom head)
2. **Custom CNN** (Built from scratch with SE attention)
3. **AST** (Audio Spectrogram Transformer - Pretrained)

### Based on Analysis:
- EfficientNet achieved **97.5% Val F1** in previous run
- AST achieved **92.9% Val F1**
- Strategy: 5-fold training + 7-step TTA

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: SETUP & IMPORTS
# ══════════════════════════════════════════════════════════════
!pip install -q transformers timm librosa wandb

import os, gc, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from transformers import ASTFeatureExtractor, ASTForAudioClassification

import wandb

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}, Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: WANDB INITIALIZATION
# ══════════════════════════════════════════════════════════════
# Initialize W&B for experiment tracking
WANDB_ENTITY = os.environ.get('WANDB_ENTITY', '23f2002518-dl-genai-project')
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', '23f2002518-t12026')
WANDB_ENABLED = bool(os.environ.get('WANDB_API_KEY'))

if WANDB_ENABLED:
    try:
        wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
        print(f"✓ W&B logged in to {WANDB_ENTITY}/{WANDB_PROJECT}")
    except Exception as e:
        WANDB_ENABLED = False
        print(f"⚠ W&B login failed: {e}")
else:
    print("⚠ WANDB_API_KEY not set; continuing without logging")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: CONFIGURATION
# ══════════════════════════════════════════════════════════════
class CFG:
    # Paths
    BASE = Path('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup')
    TRAIN_PATH = BASE / 'genres_stems'
    NOISE_PATH = BASE / 'ESC-50-master' / 'audio'
    TEST_CSV = BASE / 'test.csv'
    OUTPUT = Path('/kaggle/working')
    
    # Audio parameters
    SR = 16000
    DURATION = 10
    N_SAMPLES = SR * DURATION
    N_MELS = 128
    N_FFT = 1024
    HOP = 256
    
    # Training
    SEED = 42
    N_FOLDS = 5
    TRAIN_FOLDS = 5  # Stronger archived configuration for final submission
    
    # Model hyperparameters
    EPOCHS_EFFNET = 30
    EPOCHS_CNN = 20
    EPOCHS_AST = 20
    
    BATCH_EFFNET = 16
    BATCH_CNN = 16
    BATCH_AST = 8
    
    LR_EFFNET = 5e-4
    LR_CNN = 1e-3
    LR_AST = 5e-6
    
    # Augmentation
    MASHUP_PROB = 0.7
    NOISE_PROB = 0.5
    SNR_MIN = 8
    SNR_MAX = 28
    
    # TTA
    TTA_SHIFTS = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]  # 7 shifts
    
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
    NUM_CLASSES = 10
    NUM_WORKERS = 0  # MUST be 0 for Kaggle

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.SEED)
LE = LabelEncoder().fit(CFG.GENRES)
print(f"✓ Config ready")
print(f"  Folds: {CFG.TRAIN_FOLDS}, TTA: {len(CFG.TTA_SHIFTS)} shifts")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: LOAD DATA
# ══════════════════════════════════════════════════════════════
train_data = []
genre_songs = defaultdict(list)

for genre in CFG.GENRES:
    gpath = CFG.TRAIN_PATH / genre
    if gpath.exists():
        for song in sorted(gpath.iterdir()):
            if song.is_dir():
                train_data.append({'path': str(song), 'genre': genre})
                genre_songs[genre].append(str(song))

train_df = pd.DataFrame(train_data)
train_df['label'] = LE.transform(train_df['genre'])
print(f"✓ Train: {len(train_df)} songs ({len(train_df)//10} per genre)")

test_df = pd.read_csv(CFG.TEST_CSV)
print(f"✓ Test: {len(test_df)} samples")

noise_files = list(CFG.NOISE_PATH.glob('*.wav')) if CFG.NOISE_PATH.exists() else []
print(f"✓ Noise: {len(noise_files)} ESC-50 files")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: AUDIO PROCESSING
# ══════════════════════════════════════════════════════════════
def load_audio(path, sr=CFG.SR, duration=CFG.DURATION, offset=0.0):
    """Load and normalize audio file"""
    target = sr * duration
    try:
        y, _ = librosa.load(str(path), sr=sr, duration=duration, offset=max(0, offset))
        if len(y) < target:
            y = np.pad(y, (0, target - len(y)))
        else:
            y = y[:target]
        mx = np.max(np.abs(y))
        if mx > 1e-6:
            y = y / mx
        return y.astype(np.float32)
    except:
        return np.zeros(target, dtype=np.float32)

def load_stems(song_path):
    """Load all 4 stems from song folder"""
    stems = {}
    p = Path(song_path)
    for s in ['drums', 'vocals', 'bass', 'others']:
        sf = p / f"{s}.wav"
        stems[s] = load_audio(sf) if sf.exists() else np.zeros(CFG.N_SAMPLES, dtype=np.float32)
    return stems

def create_mashup(genre, genre_songs):
    """Create synthetic mashup from different songs"""
    songs = genre_songs[genre]
    if len(songs) < 4:
        stems = load_stems(random.choice(songs))
        w = np.random.uniform(0.5, 1.5, 4)
        m = sum(s * wt for s, wt in zip(stems.values(), w))
    else:
        sel = random.sample(songs, 4)
        m = np.zeros(CFG.N_SAMPLES, dtype=np.float32)
        for i, sn in enumerate(['drums', 'vocals', 'bass', 'others']):
            sp = Path(sel[i]) / f"{sn}.wav"
            if sp.exists():
                m += np.random.uniform(0.5, 1.5) * load_audio(sp)
    mx = np.max(np.abs(m))
    return (m / mx if mx > 1e-6 else m).astype(np.float32)

def add_noise(audio, noise_files):
    """Add ESC-50 environmental noise"""
    if not noise_files or random.random() > CFG.NOISE_PROB:
        return audio
    snr = np.random.uniform(CFG.SNR_MIN, CFG.SNR_MAX)
    nf = random.choice(noise_files)
    try:
        noise, _ = librosa.load(str(nf), sr=CFG.SR)
        while len(noise) < len(audio):
            noise = np.tile(noise, 2)
        st = random.randint(0, max(0, len(noise) - len(audio)))
        noise = noise[st:st+len(audio)]
        if len(noise) < len(audio):
            noise = np.pad(noise, (0, len(audio) - len(noise)))
        ap = np.mean(audio**2) + 1e-10
        np_ = np.mean(noise**2) + 1e-10
        scale = np.sqrt(ap / (10**(snr/10) * np_))
        m = audio + scale * noise
        mx = np.max(np.abs(m))
        return (m / mx if mx > 1e-6 else m).astype(np.float32)
    except:
        return audio

def to_mel(y):
    """Convert to log mel spectrogram"""
    mel = librosa.feature.melspectrogram(y=y, sr=CFG.SR, n_mels=CFG.N_MELS, n_fft=CFG.N_FFT, hop_length=CFG.HOP)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return ((mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)).astype(np.float32)

print("✓ Audio functions defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: DATASETS
# ══════════════════════════════════════════════════════════════
class MelTrainDataset(Dataset):
    def __init__(self, df, genre_songs, noise_files):
        self.df = df.reset_index(drop=True)
        self.genre_songs = genre_songs
        self.noise_files = noise_files
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if random.random() < CFG.MASHUP_PROB:
            audio = create_mashup(row['genre'], self.genre_songs)
        else:
            stems = load_stems(row['path'])
            w = np.random.uniform(0.5, 1.5, 4)
            audio = sum(s * wt for s, wt in zip(stems.values(), w))
            mx = np.max(np.abs(audio))
            if mx > 1e-6: audio = audio / mx
        audio = add_noise(audio, self.noise_files)
        return torch.from_numpy(to_mel(audio)).unsqueeze(0), torch.tensor(row['label'], dtype=torch.long)

class MelValDataset(Dataset):
    def __init__(self, df, genre_songs):
        self.df = df.reset_index(drop=True)
        self.genre_songs = genre_songs
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        stems = load_stems(row['path'])
        audio = sum(stems.values())
        mx = np.max(np.abs(audio))
        if mx > 1e-6: audio = audio / mx
        return torch.from_numpy(to_mel(audio)).unsqueeze(0), torch.tensor(row['label'], dtype=torch.long)

class TestMelDataset(Dataset):
    def __init__(self, df, offset=0.0):
        self.df = df.reset_index(drop=True)
        self.offset = offset
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = load_audio(CFG.BASE / row['filename'], offset=self.offset)
        return torch.from_numpy(to_mel(audio)).unsqueeze(0), row['id']

class ASTTrainDataset(Dataset):
    def __init__(self, df, genre_songs, noise_files, fe):
        self.df = df.reset_index(drop=True)
        self.genre_songs = genre_songs
        self.noise_files = noise_files
        self.fe = fe
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if random.random() < CFG.MASHUP_PROB:
            audio = create_mashup(row['genre'], self.genre_songs)
        else:
            stems = load_stems(row['path'])
            w = np.random.uniform(0.5, 1.5, 4)
            audio = sum(s * wt for s, wt in zip(stems.values(), w))
            mx = np.max(np.abs(audio))
            if mx > 1e-6: audio = audio / mx
        audio = add_noise(audio, self.noise_files)
        inputs = self.fe(audio, sampling_rate=CFG.SR, return_tensors="pt", padding="max_length", max_length=1024)
        return inputs.input_values.squeeze(0), torch.tensor(row['label'], dtype=torch.long)

class ASTValDataset(Dataset):
    def __init__(self, df, genre_songs, fe):
        self.df = df.reset_index(drop=True)
        self.genre_songs = genre_songs
        self.fe = fe
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        stems = load_stems(row['path'])
        audio = sum(stems.values())
        mx = np.max(np.abs(audio))
        if mx > 1e-6: audio = audio / mx
        inputs = self.fe(audio, sampling_rate=CFG.SR, return_tensors="pt", padding="max_length", max_length=1024)
        return inputs.input_values.squeeze(0), torch.tensor(row['label'], dtype=torch.long)

class ASTTestDataset(Dataset):
    def __init__(self, df, fe, offset=0.0):
        self.df = df.reset_index(drop=True)
        self.fe = fe
        self.offset = offset
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = load_audio(CFG.BASE / row['filename'], offset=self.offset)
        inputs = self.fe(audio, sampling_rate=CFG.SR, return_tensors="pt", padding="max_length", max_length=1024)
        return inputs.input_values.squeeze(0), row['id']

print("✓ Datasets defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7: MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════

# Model 1: EfficientNet-B0 (Pretrained backbone)
class EfficientNetClassifier(nn.Module):
    """EfficientNet-B0 pretrained on ImageNet, adapted for spectrograms"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, in_chans=1, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.head(self.backbone(x))

# Model 2: Custom CNN from scratch with SE attention
class SEBlock(nn.Module):
    """Squeeze-and-Excitation attention block"""
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch//r), nn.ReLU(),
            nn.Linear(ch//r, ch), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)

class ConvBlock(nn.Module):
    """Conv block with SE attention"""
    def __init__(self, inc, outc):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(inc, outc, 3, padding=1), nn.BatchNorm2d(outc), nn.ReLU(),
            nn.Conv2d(outc, outc, 3, padding=1), nn.BatchNorm2d(outc), nn.ReLU(),
            SEBlock(outc), nn.MaxPool2d(2)
        )
    def forward(self, x):
        return self.conv(x)

class CustomCNN(nn.Module):
    """Custom CNN built from scratch with SE attention blocks"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.fc(self.pool(self.features(x)))

# Model 3: AST Wrapper
class ASTWrapper(nn.Module):
    """Wrapper for HuggingFace AST model"""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x).logits

# Test models
x = torch.randn(2, 1, 128, 626)
m1 = EfficientNetClassifier()
m2 = CustomCNN()
print(f"EfficientNet: {sum(p.numel() for p in m1.parameters()):,} params")
print(f"CustomCNN: {sum(p.numel() for p in m2.parameters()):,} params")
del m1, m2, x

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses, preds, labels = [], [], []
    for feat, lab in tqdm(loader, leave=False):
        feat, lab = feat.to(device), lab.to(device)
        optimizer.zero_grad()
        out = model(feat)
        loss = criterion(out, lab)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lab.cpu().numpy())
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return np.mean(losses), f1, acc

@torch.no_grad()
def valid_epoch(model, loader, criterion, device):
    model.eval()
    losses, preds, labels = [], [], []
    for feat, lab in tqdm(loader, leave=False):
        feat, lab = feat.to(device), lab.to(device)
        out = model(feat)
        loss = criterion(out, lab)
        losses.append(loss.item())
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lab.cpu().numpy())
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return np.mean(losses), f1, acc

@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    all_probs, all_ids = [], []
    for feat, ids in tqdm(loader, leave=False):
        feat = feat.to(device)
        probs = F.softmax(model(feat), dim=1).cpu().numpy()
        all_probs.extend(probs)
        for i in ids:
            all_ids.append(i.item() if isinstance(i, torch.Tensor) else i)
    return np.array(all_probs), all_ids

print("✓ Training utilities")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: TRAIN EFFICIENTNET (Pretrained)
# ══════════════════════════════════════════════════════════════
print("="*60)
print("MODEL 1: EFFICIENTNET-B0 (Pretrained)")
print("="*60)

# Initialize W&B run for EfficientNet
if WANDB_ENABLED:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="efficientnet-b0", 
               config={"model": "efficientnet_b0", "epochs": CFG.EPOCHS_EFFNET, 
                       "lr": CFG.LR_EFFNET, "batch_size": CFG.BATCH_EFFNET})

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
effnet_models = []
effnet_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    if fold >= CFG.TRAIN_FOLDS:
        break
    
    print(f"\n--- Fold {fold+1}/{CFG.TRAIN_FOLDS} ---")
    
    train_ds = MelTrainDataset(train_df.iloc[train_idx], genre_songs, noise_files)
    val_ds = MelValDataset(train_df.iloc[val_idx], genre_songs)
    train_ld = DataLoader(train_ds, CFG.BATCH_EFFNET, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_ld = DataLoader(val_ds, CFG.BATCH_EFFNET, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    
    model = EfficientNetClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=CFG.LR_EFFNET, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, CFG.EPOCHS_EFFNET)
    
    best_f1 = 0
    patience, no_imp = 7, 0
    
    for ep in range(CFG.EPOCHS_EFFNET):
        tl, tf, ta = train_epoch(model, train_ld, criterion, optimizer, DEVICE)
        vl, vf, va = valid_epoch(model, val_ld, criterion, DEVICE)
        scheduler.step()
        
        print(f"Ep{ep+1:2d}: TF1={tf:.4f} TAcc={ta:.4f} | VF1={vf:.4f} VAcc={va:.4f}")
        
        # Log to W&B
        if WANDB_ENABLED:
            wandb.log({f"fold{fold}/train_loss": tl, f"fold{fold}/train_f1": tf, f"fold{fold}/train_acc": ta,
                       f"fold{fold}/val_loss": vl, f"fold{fold}/val_f1": vf, f"fold{fold}/val_acc": va})
        
        if vf > best_f1:
            best_f1 = vf
            torch.save(model.state_dict(), CFG.OUTPUT / f'effnet_f{fold}.pt')
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"Early stop at ep{ep+1}")
                break
    
    model.load_state_dict(torch.load(CFG.OUTPUT / f'effnet_f{fold}.pt', weights_only=True))
    effnet_models.append(model)
    effnet_scores.append(best_f1)
    print(f"✓ Fold {fold+1} Best F1: {best_f1:.4f}")
    gc.collect(); torch.cuda.empty_cache()

if WANDB_ENABLED:
    wandb.log({"effnet_mean_f1": np.mean(effnet_scores), "effnet_std_f1": np.std(effnet_scores)})
    wandb.finish()

print(f"\n✓ EfficientNet: {np.mean(effnet_scores):.4f} ± {np.std(effnet_scores):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10: TRAIN CUSTOM CNN (From Scratch)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("MODEL 2: CUSTOM CNN (From Scratch)")
print("="*60)

if WANDB_ENABLED:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="custom-cnn",
               config={"model": "custom_cnn_se", "epochs": CFG.EPOCHS_CNN,
                       "lr": CFG.LR_CNN, "batch_size": CFG.BATCH_CNN})

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
cnn_models = []
cnn_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    if fold >= 1:  # Only 1 fold for scratch model (takes longer to train)
        break
    
    print(f"\n--- Fold {fold+1} ---")
    
    train_ds = MelTrainDataset(train_df.iloc[train_idx], genre_songs, noise_files)
    val_ds = MelValDataset(train_df.iloc[val_idx], genre_songs)
    train_ld = DataLoader(train_ds, CFG.BATCH_CNN, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_ld = DataLoader(val_ds, CFG.BATCH_CNN, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    
    model = CustomCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=CFG.LR_CNN, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, CFG.EPOCHS_CNN)
    
    best_f1 = 0
    patience, no_imp = 10, 0
    
    for ep in range(CFG.EPOCHS_CNN):
        tl, tf, ta = train_epoch(model, train_ld, criterion, optimizer, DEVICE)
        vl, vf, va = valid_epoch(model, val_ld, criterion, DEVICE)
        scheduler.step()
        
        print(f"Ep{ep+1:2d}: TF1={tf:.4f} TAcc={ta:.4f} | VF1={vf:.4f} VAcc={va:.4f}")
        
        if WANDB_ENABLED:
            wandb.log({"train_loss": tl, "train_f1": tf, "train_acc": ta,
                       "val_loss": vl, "val_f1": vf, "val_acc": va})
        
        if vf > best_f1:
            best_f1 = vf
            torch.save(model.state_dict(), CFG.OUTPUT / f'cnn_f{fold}.pt')
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"Early stop at ep{ep+1}")
                break
    
    model.load_state_dict(torch.load(CFG.OUTPUT / f'cnn_f{fold}.pt', weights_only=True))
    cnn_models.append(model)
    cnn_scores.append(best_f1)
    print(f"✓ CustomCNN Best F1: {best_f1:.4f}")
    gc.collect(); torch.cuda.empty_cache()

if WANDB_ENABLED:
    wandb.log({"cnn_best_f1": best_f1})
    wandb.finish()

print(f"\n✓ CustomCNN: {np.mean(cnn_scores):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11: TRAIN AST (Pretrained Transformer)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("MODEL 3: AST (Audio Spectrogram Transformer)")
print("="*60)

ast_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
print(f"Loading: {ast_name}")
ast_fe = ASTFeatureExtractor.from_pretrained(ast_name)

if WANDB_ENABLED:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="ast-transformer",
               config={"model": "ast-finetuned-audioset", "epochs": CFG.EPOCHS_AST,
                       "lr": CFG.LR_AST, "batch_size": CFG.BATCH_AST})

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
ast_models = []
ast_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    if fold >= CFG.TRAIN_FOLDS:
        break
    
    print(f"\n--- Fold {fold+1}/{CFG.TRAIN_FOLDS} ---")
    
    train_ds = ASTTrainDataset(train_df.iloc[train_idx], genre_songs, noise_files, ast_fe)
    val_ds = ASTValDataset(train_df.iloc[val_idx], genre_songs, ast_fe)
    train_ld = DataLoader(train_ds, CFG.BATCH_AST, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_ld = DataLoader(val_ds, CFG.BATCH_AST, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    
    ast_base = ASTForAudioClassification.from_pretrained(
        ast_name, num_labels=CFG.NUM_CLASSES, ignore_mismatched_sizes=True
    )
    model = ASTWrapper(ast_base).to(DEVICE)
    
    # Freeze embeddings
    for param in ast_base.audio_spectrogram_transformer.embeddings.parameters():
        param.requires_grad = False
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CFG.LR_AST, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, CFG.EPOCHS_AST)
    
    best_f1 = 0
    patience, no_imp = 5, 0
    
    for ep in range(CFG.EPOCHS_AST):
        tl, tf, ta = train_epoch(model, train_ld, criterion, optimizer, DEVICE)
        vl, vf, va = valid_epoch(model, val_ld, criterion, DEVICE)
        scheduler.step()
        
        print(f"Ep{ep+1:2d}: TF1={tf:.4f} TAcc={ta:.4f} | VF1={vf:.4f} VAcc={va:.4f}")
        
        if WANDB_ENABLED:
            wandb.log({f"fold{fold}/train_loss": tl, f"fold{fold}/train_f1": tf, f"fold{fold}/train_acc": ta,
                       f"fold{fold}/val_loss": vl, f"fold{fold}/val_f1": vf, f"fold{fold}/val_acc": va})
        
        if vf > best_f1:
            best_f1 = vf
            torch.save(model.state_dict(), CFG.OUTPUT / f'ast_f{fold}.pt')
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"Early stop at ep{ep+1}")
                break
    
    model.load_state_dict(torch.load(CFG.OUTPUT / f'ast_f{fold}.pt', weights_only=True))
    ast_models.append(model)
    ast_scores.append(best_f1)
    print(f"✓ Fold {fold+1} Best F1: {best_f1:.4f}")
    gc.collect(); torch.cuda.empty_cache()

if WANDB_ENABLED:
    wandb.log({"ast_mean_f1": np.mean(ast_scores), "ast_std_f1": np.std(ast_scores)})
    wandb.finish()

print(f"\n✓ AST: {np.mean(ast_scores):.4f} ± {np.std(ast_scores):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12: MODEL COMPARISON
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

print(f"\n{'Model':<20} {'Val F1':>10} {'Folds':>8}")
print("-"*40)
print(f"{'EfficientNet-B0':<20} {np.mean(effnet_scores):>10.4f} {len(effnet_scores):>8}")
print(f"{'CustomCNN (scratch)':<20} {np.mean(cnn_scores):>10.4f} {len(cnn_scores):>8}")
print(f"{'AST Transformer':<20} {np.mean(ast_scores):>10.4f} {len(ast_scores):>8}")

# Log comparison to W&B
if WANDB_ENABLED:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="model-comparison")
    wandb.log({
        "comparison/efficientnet_f1": np.mean(effnet_scores),
        "comparison/customcnn_f1": np.mean(cnn_scores),
        "comparison/ast_f1": np.mean(ast_scores)
    })
    wandb.finish()

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13: INFERENCE WITH TTA
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("INFERENCE WITH TTA")
print("="*60)

test_df = pd.read_csv(CFG.TEST_CSV)
print(f"Test: {len(test_df)}, TTA: {CFG.TTA_SHIFTS}")

all_preds = []
all_weights = []
file_ids = None

# EfficientNet
print(f"\n▶ EfficientNet ({len(effnet_models)} models × {len(CFG.TTA_SHIFTS)} TTA)")
for m_idx, model in enumerate(effnet_models):
    for shift in CFG.TTA_SHIFTS:
        test_ds = TestMelDataset(test_df, offset=shift)
        test_ld = DataLoader(test_ds, CFG.BATCH_EFFNET, shuffle=False, num_workers=CFG.NUM_WORKERS)
        probs, ids = predict_proba(model, test_ld, DEVICE)
        all_preds.append(probs)
        all_weights.append(effnet_scores[m_idx] * 1.5)  # Higher weight for best model
        if file_ids is None:
            file_ids = ids
    print(f"  Model {m_idx+1}: done")

# CustomCNN (lower weight - typically lower performance)
print(f"\n▶ CustomCNN ({len(cnn_models)} models × {len(CFG.TTA_SHIFTS)} TTA)")
for m_idx, model in enumerate(cnn_models):
    for shift in CFG.TTA_SHIFTS:
        test_ds = TestMelDataset(test_df, offset=shift)
        test_ld = DataLoader(test_ds, CFG.BATCH_CNN, shuffle=False, num_workers=CFG.NUM_WORKERS)
        probs, _ = predict_proba(model, test_ld, DEVICE)
        all_preds.append(probs)
        all_weights.append(cnn_scores[m_idx] * 0.5)  # Lower weight
    print(f"  Model {m_idx+1}: done")

# AST
print(f"\n▶ AST ({len(ast_models)} models × {len(CFG.TTA_SHIFTS)} TTA)")
for m_idx, model in enumerate(ast_models):
    for shift in CFG.TTA_SHIFTS:
        test_ds = ASTTestDataset(test_df, ast_fe, offset=shift)
        test_ld = DataLoader(test_ds, CFG.BATCH_AST, shuffle=False, num_workers=CFG.NUM_WORKERS)
        probs, _ = predict_proba(model, test_ld, DEVICE)
        all_preds.append(probs)
        all_weights.append(ast_scores[m_idx] * 1.0)
    print(f"  Model {m_idx+1}: done")

print(f"\n✓ Total: {len(all_preds)} prediction sets")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14: ENSEMBLE & SUBMISSION
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ENSEMBLE & SUBMISSION")
print("="*60)

# Normalize weights
weights = np.array(all_weights)
weights = weights / weights.sum()

# Weighted ensemble
ensemble = np.zeros_like(all_preds[0])
for w, p in zip(weights, all_preds):
    ensemble += w * p

final_preds = np.argmax(ensemble, axis=1)
final_genres = LE.inverse_transform(final_preds)

# Create submission
submission = pd.DataFrame({'id': file_ids, 'genre': final_genres})
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\n✓ Saved: submission.csv ({len(submission)} rows)")
print(f"\nFirst 10:")
print(submission.head(10).to_string(index=False))

print(f"\nDistribution:")
for g, c in submission['genre'].value_counts().sort_index().items():
    print(f"  {g:12s}: {c:4d} ({c/len(submission)*100:5.1f}%)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 15: VERIFICATION & SUMMARY
# ══════════════════════════════════════════════════════════════
import os

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

if os.path.exists('/kaggle/working/submission.csv'):
    print("\n✅ submission.csv EXISTS")
    sub = pd.read_csv('/kaggle/working/submission.csv')
    print(f"Rows: {len(sub)}, Expected: {len(test_df)}")
    print(f"✅ Row count {'CORRECT' if len(sub) == len(test_df) else 'WRONG'}")
else:
    print("❌ submission.csv NOT FOUND")

print(f"\nFiles:")
for f in sorted(os.listdir('/kaggle/working/')):
    sz = os.path.getsize(f'/kaggle/working/{f}')
    print(f"  {f}: {sz:,} bytes")

print(f"\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\n3 Models Trained:")
print(f"  1. EfficientNet-B0 (pretrained): {np.mean(effnet_scores):.4f} F1")
print(f"  2. CustomCNN (from scratch): {np.mean(cnn_scores):.4f} F1")
print(f"  3. AST Transformer (pretrained): {np.mean(ast_scores):.4f} F1")
print(f"\nEnsemble: {len(all_preds)} predictions with TTA")
print(f"\n✓ Ready for Kaggle submission!")